In [1]:
import os
import glob
import collections
import xarray as xr
import numpy as np
import pandas as pd
from datetime import datetime
from skimage.feature import peak_local_max
import matplotlib.pyplot as plt
import matplotlib.patheffects as PathEffects
import matplotlib.colors as mcolors
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import cartopy.mpl.ticker as cticker
import regionmask

from scipy import stats
from scipy.stats import pearsonr
from scipy.signal import butter, filtfilt, sosfilt, lfilter
from matplotlib.pylab import rcParams
from matplotlib.patches import Polygon


/home/ac.szhang/.conda/envs/zppy-pcmdi-diags/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
ERROR 1: PROJ: proj_create_from_database: Open of /home/ac.szhang/.conda/envs/zppy-pcmdi-diags/share/proj failed


In [3]:
# ============================================================
# Helper: Pearson r p-value (replaces xskillscore)
# ============================================================
def pearson_r_p_value(a, b, dim, skipna=True):
    """Two-tailed p-value of Pearson correlation over dimension `dim`."""
    n = a.sizes[dim]
    r = xr.corr(a, b, dim=dim)
    t_stat = r * np.sqrt((n - 2) / (1 - r**2))
    pval = 2 * stats.t.sf(np.abs(t_stat.values), df=n - 2)
    return xr.DataArray(pval, dims=r.dims, coords=r.coords)


# ============================================================
# Helper: Blue-Yellow-Red color list (replaces cmaps + geocat)
# ============================================================
def get_byr_colorlist(n=20):
    """Return a list of n colors approximating NCL's BlueYellowRed (white at center)."""
    cmap = plt.cm.RdYlBu_r  # built-in Blue->Yellow->Red diverging colormap
    colors = [list(cmap(i / (n - 1))) for i in range(n)]
    mid = n // 2
    colors[mid - 1] = [1., 1., 1., 1.]
    colors[mid    ] = [1., 1., 1., 1.]
    return colors


# ============================================================
# Low-pass filter
# ============================================================
def low_pass(cutoff_freq, data, order=5, axis=-1):
    Wn = cutoff_freq
    b, a = butter(order, Wn, btype='lowpass', analog=False)
    data_filt = filtfilt(b, a, data, axis=axis, method="gust")
    return data_filt


# ============================================================
# Detrend along a dimension
# ============================================================
def detrend_dim(da, dim, deg=1):
    p = da.polyfit(dim=dim, deg=deg)
    fit = xr.polyval(da[dim], p.polyfit_coefficients)
    return da - fit


# ============================================================
# Slice region
# ============================================================
def slice_region(da, region, boarder=8):
    latn = region['north'] + boarder
    lats = region['south'] - boarder
    lone = region['east']  + boarder
    lonw = region['west']  - boarder
    da = da.sel(latitude=slice(lats, latn), longitude=slice(lonw, lone))
    return da


# ============================================================
# ASL sector mean
# ============================================================
def asl_sector_mean(da, region):
    latn = region['north']
    lats = region['south']
    lone = region['east']
    lonw = region['west']
    a = da.sel(latitude=slice(lats, latn),
               longitude=slice(lonw, lone)).mean().values
    return a


# ============================================================
# Find pressure lows
# ============================================================
def get_lows(da, asl_region, min_dist, num_peak, exclue_border):
    import scipy.ndimage as ndimage

    lons, lats = da.longitude.values, da.latitude.values
    sector_mean_pres = asl_sector_mean(da, asl_region)
    threshold = sector_mean_pres
    time_str = str(da.time.values)[:10]
    da_max = da.max().values
    da = da.fillna(da_max)
    invert_data = (da * -1.).values

    if threshold is None:
        threshold_abs = invert_data.mean()
    else:
        threshold_abs = threshold * -1

    minima_yx = peak_local_max(invert_data,
                               min_distance=min_dist,
                               num_peaks=num_peak,
                               exclude_border=exclue_border,
                               threshold_abs=threshold_abs)
    minima_lat, minima_lon, pressure = [], [], []
    for minima in minima_yx:
        minima_lat.append(lats[minima[0]])
        minima_lon.append(lons[minima[1]])
        pressure.append(da.values[minima[0], minima[1]])

    df = pd.DataFrame()
    df['lat']        = minima_lat
    df['lon']        = minima_lon
    df['ActCenPres'] = pressure
    df['SectorPres'] = sector_mean_pres
    df['time']       = time_str
    df['RelCenPres'] = df['ActCenPres'] - df['SectorPres']
    df = df[['time', 'lon', 'lat', 'ActCenPres', 'SectorPres', 'RelCenPres']]
    df = df.reset_index(drop=True)
    return df


# ============================================================
# Define ASL from detected lows
# ============================================================
def define_asl(times, df, region, l_allow_no_asl, mip, exp, relm, case, case_id, period, vstr, out_path):
    df2 = df[(df['lon'] > region['west'])  &
             (df['lon'] < region['east'])  &
             (df['lat'] > region['south']) &
             (df['lat'] < region['north'])]
    df2 = df2.loc[df2.groupby('time')['ActCenPres'].idxmin()]
    df2 = df2.reset_index(drop=True)

    if len(df2) != len(times):
        print("WARNNING from define_asl()")
        print("length of time in data: ", len(times))
        print("length of time with identified ASL: ", len(df2))
        print("there are missing time steps in asl data, can not continue...")
        c = list(set(df2['time']).symmetric_difference(times.values))
        print("no asl present in following months:", c)
        if l_allow_no_asl:
            for time_str in c:
                asl_df               = pd.DataFrame()
                asl_df['lat']        = [np.nan]
                asl_df['lon']        = [np.nan]
                asl_df['ActCenPres'] = [np.nan]
                asl_df['SectorPres'] = [np.nan]
                asl_df['RelCenPres'] = [np.nan]
                asl_df['time']       = time_str
                df2 = pd.concat([df2, asl_df], ignore_index=True)
                del asl_df
        else:
            df1 = df.loc[df.groupby('time')['ActCenPres'].idxmin()]
            df1 = df1.reset_index(drop=True)
            for time_str in c:
                indx = list(df1['time']).index(time_str)
                asl_df               = pd.DataFrame()
                asl_df['lat']        = [df1['lat'][indx]]
                asl_df['lon']        = [df1['lon'][indx]]
                asl_df['ActCenPres'] = [df1['ActCenPres'][indx]]
                asl_df['SectorPres'] = [df1['SectorPres'][indx]]
                asl_df['RelCenPres'] = [df1['RelCenPres'][indx]]
                asl_df['time']       = time_str
                df2 = pd.concat([df2, asl_df], sort=True)
                del asl_df, indx
            del df1

    colnams = ['time', 'lon', 'lat', 'ActCenPres', 'SectorPres', 'RelCenPres']
    df2 = df2.sort_values(by="time")
    df2 = df2.reset_index(drop=True)
    df2 = df2[colnams]

    if not os.path.exists(out_path):
        os.makedirs(out_path)
    out_file = '{}.{}.{}.{}.{}.{}.{}.csv'.format(mip, exp, case, relm, case_id, vstr, period)
    df2.to_csv(os.path.join(out_path, out_file), index=False)
    return df2, colnams


# ============================================================
# Save regression data to netCDF
# ============================================================
def save_figure_data(vcor, vreg, pval, vsig, mip, exp, relm, case, case_id, period, var, vunt, out_path):
    lons, lats = vcor.longitude, vcor.latitude
    df = xr.Dataset(
        {
            "var_reg": (["lat", "lon"], vreg.data),
            "var_sig": (["lat", "lon"], vsig.data),
            "var_cor": (["lat", "lon"], vcor.data),
            "pval":    (["lat", "lon"], pval.data),
        },
        coords={
            "lon": (["lon"], lons.data),
            "lat": (["lat"], lats.data),
        },
        attrs=dict(description="sam regression maps", reference_time=period),
    )
    df.lat.attrs["units"]         = "degree_north"
    df.lon.attrs["units"]         = "degree_east"
    df.var_cor.attrs["units"]     = "1"
    df.var_cor.attrs["long_name"] = var + " correlation"
    df.var_reg.attrs["units"]     = vunt
    df.var_reg.attrs["long_name"] = var + " regression"
    df.var_sig.attrs["units"]     = vunt
    df.var_sig.attrs["long_name"] = var + " regression (significant at 0.05 confidence level)"
    df.pval.attrs["units"]        = "1"
    df.pval.attrs["long_name"]    = "p values"

    if not os.path.exists(out_path):
        os.makedirs(out_path)
    fout_name = "{}_{}_{}_{}_{}_{}_{}.nc".format(mip, exp, case, relm, case_id, var, period)
    df.to_netcdf(os.path.join(out_path, fout_name))
    return


# ============================================================
# Draw regional box on a map
# ============================================================
def draw_regional_box(region, transform=None):
    if transform is None:
        transform = ccrs.PlateCarree()
    plt.plot([region['west'], region['west']], [region['south'], region['north']],
             'k-', transform=transform, linewidth=1)
    plt.plot([region['east'], region['east']], [region['south'], region['north']],
             'k-', transform=transform, linewidth=1)
    for i in range(np.int32(region['west']), np.int32(region['east'])):
        plt.plot([i, i+1], [region['south'], region['south']], 'k-', transform=transform, linewidth=1)
        plt.plot([i, i+1], [region['north'], region['north']], 'k-', transform=transform, linewidth=1)


# ============================================================
# Plot region sanity check
# ============================================================
def plot_regions_mask(fig_path, da, da_mask, group):
    if not os.path.exists(fig_path):
        os.makedirs(fig_path)
    plt.figure(figsize=(5, 5))
    ax1 = plt.subplot(121, projection=ccrs.Stereographic(central_longitude=0., central_latitude=-90.))
    ax1.set_extent([-180, 180, -90, -50], ccrs.PlateCarree())
    da.isel(time=-1).plot.pcolormesh('longitude', 'latitude', cmap='jet',
                                     transform=ccrs.PlateCarree(), add_colorbar=False)
    ax2 = plt.subplot(122, projection=ccrs.Stereographic(central_longitude=0., central_latitude=-90.))
    ax2.set_extent([-180, 180, -90, -50], ccrs.PlateCarree())
    da_mask.isel(time=-1).plot.pcolormesh('longitude', 'latitude', cmap='jet',
                                          transform=ccrs.PlateCarree(), add_colorbar=False)
    plt.savefig(os.path.join(fig_path, "asl_region_{}.pdf".format(group)))
    plt.close()
    return


# ============================================================
# Draw ASL time series
# ============================================================
def draw_asl_ts(fig_path, asl_df, colnams, asl_region, mip, exp, relm, case, case_id, period, var, vunt):
    time  = asl_df['time']
    xtime = np.linspace(1, len(time), len(time))
    years = int(time[0].split("-")[0])
    yeare = int(time[len(time)-1].split("-")[0])
    xtick = np.arange(0, len(time))
    xlabs = np.arange(0, len(time)) / 12.0 + years
    fontsize = 16
    fig = plt.figure(figsize=(8, 11))
    for i, col in enumerate(colnams[1:]):
        print("plot {}".format(col))
        var0 = np.array(asl_df[col])
        var1 = low_pass(1.0/11.0, var0, axis=0)
        ax = fig.add_subplot(len(colnams), 1, i+1)
        ax.plot(xtime, var0, color='grey', alpha=1.0, linewidth=0.8, label='monthly')
        ax.plot(xtime, var1, color='black', alpha=1.0, linewidth=1.4, label='11-point Hamming')
        ax.set_xlabel('Time (years)')
        ax.set_ylabel(col)
        ax.set_xlim(0, len(time))
        ax.set_xticks(xtick[::120])
        ax.set_xticklabels(xlabs[::120].astype(int))
        ax.grid(True)
        if i+1 == len(colnams):
            ax.legend(loc='lower right', prop={'size': 8})
        del var0, var1
    plt.suptitle('ASL indices ({},{})'.format(var, vunt), fontsize=fontsize*1.1)
    plt.tight_layout()
    if not os.path.exists(fig_path):
        os.makedirs(fig_path)
    fig_name = "fig_ts_{}_{}_{}_{}_{}_{}_{}.pdf".format(mip, exp, case, relm, case_id, var, period)
    plt.savefig(os.path.join(fig_path, fig_name))
    plt.close()
    return


# ============================================================
# Draw ASL location map
# ============================================================
def draw_asl_loc(da, mask, asl_df, asl_region, mip, exp, relm, case, case_id, period, vstr, vunt, fig_path):
    da_nmsk = slice_region(da, asl_region)
    da_mask = da.where(mask == 0)
    da_mask = slice_region(da_mask, asl_region)
    plt.figure(figsize=(20, 15))
    for i in range(0, 12):
        da_2D = da_mask.isel(time=i)
        da_2D = da_2D.sel(latitude=slice(-90, -55), longitude=slice(165, 305))
        ax = plt.subplot(3, 4, i+1,
                         projection=ccrs.Stereographic(central_longitude=0., central_latitude=-90.))
        ax.set_extent([165, 305, -85, -55], ccrs.PlateCarree())
        result = da_2D.plot.contourf('longitude', 'latitude', cmap='Reds',
                                     transform=ccrs.PlateCarree(),
                                     add_colorbar=False,
                                     levels=np.linspace(np.nanmin(da_2D.values),
                                                        np.nanmax(da_2D.values), 20))
        ax.set_title('{}({}): {}'.format(vstr, vunt, str(da_2D.time.values)[0:7]))
        df2 = asl_df[asl_df['time'] == str(da_2D.time.values)[0:10]]
        if len(df2) > 0:
            ax.plot(df2['lon'], df2['lat'], 'mx', transform=ccrs.PlateCarree())
        draw_regional_box(asl_region)
    if not os.path.exists(fig_path):
        os.makedirs(fig_path)
    fig_name = "fig_aslloc_map_{}_{}_{}_{}_{}_{}_1-12month.pdf".format(
                mip, exp, case, relm, case_id, vstr)
    plt.savefig(os.path.join(fig_path, fig_name))
    plt.close()
    del da_nmsk, da_mask
    return


# ============================================================
# Draw regression map  (replaces geocat.viz.util + cmaps)
# ============================================================
def draw_regression_map(vstr, vunt, lats, lons, cor, reg, pval, fig, region,
                        fontsize, title, grid_space, vmin, vmax, nlev):
    sig    = pval.copy()
    sig[:] = 1.0 - sig[:]
    t90 = 0.94
    t95 = 0.95
    rlabel = '{}({})'.format(vstr, vunt)

    ax = fig.add_subplot(grid_space,
                         projection=ccrs.PlateCarree(central_longitude=210))
    ax.coastlines(linewidth=0.5, alpha=0.6)

    # Set axes limits and ticks (replaces gvutil.set_axes_limits_and_ticks)
    ax.set_xlim(-180, 180)
    ax.set_ylim(-90, 90)
    ax.set_xticks(np.arange(-180, 181, 60), crs=ccrs.PlateCarree())
    ax.set_yticks(np.arange(-90, 91, 30), crs=ccrs.PlateCarree())

    # Add lat/lon tick labels (replaces gvutil.add_lat_lon_ticklabels)
    ax.xaxis.set_major_formatter(cticker.LongitudeFormatter())
    ax.yaxis.set_major_formatter(cticker.LatitudeFormatter())

    # Set tick label size (replaces gvutil.add_major_minor_ticks)
    ax.tick_params(labelsize=fontsize * 0.90)

    # Build color list (replaces gvcmaps.BlueYellowRed)
    color_list = get_byr_colorlist(nlev - 1)

    kwargs = dict(
        vmin=vmin,
        vmax=vmax,
        levels=nlev,
        colors=color_list,
        add_colorbar=False,
        transform=ccrs.PlateCarree(),
    )
    fillplot = cor.plot.contourf(ax=ax, **kwargs)

    ax.add_feature(cfeature.LAND, facecolor='lightgray', zorder=1)
    ax.add_feature(cfeature.COASTLINE, edgecolor='gray', linewidth=0.5, zorder=1)

    sig.plot.contourf(ax=ax, levels=[-1*t95, -1*t90, t90, t95], colors='none',
                      hatches=[None, None, None, '..', '..'], extend='both',
                      add_colorbar=False, transform=ccrs.PlateCarree())

    delc = 0.2
    levels = np.arange(-3, 0, delc)
    levels = np.append(levels, np.arange(delc, 3, delc))
    rad = reg.plot.contour(ax=ax, colors='black', alpha=0.8, linewidths=1.0,
                           add_labels=False, levels=levels, transform=ccrs.PlateCarree())
    pe = [PathEffects.withStroke(linewidth=2.0, foreground="w")]
    # Use .lines instead of .collections for contour line objects
    if hasattr(rad, 'lines'):
        plt.setp(rad.lines, path_effects=pe)
    elif hasattr(rad, 'collections'):
        plt.setp(rad.collections, path_effects=pe)

    # Set titles and labels (replaces gvutil.set_titles_and_labels)
    ax.set_title(title,  loc='left',  fontsize=fontsize * 0.95)
    ax.set_title(rlabel, loc='right', fontsize=fontsize * 0.95)
    ax.set_xlabel("")
    ax.set_ylabel("")

    draw_regional_box(region)
    ax.xaxis.tick_bottom()
    ax.yaxis.tick_left()
    return ax, fillplot


# ============================================================
# Draw ASL regression map
# ============================================================
def draw_asl_map(out_path, fig_path, da, asl_df, colnams, asl_region,
                 mip, exp, relm, case, case_id, period, var, vunt):
    lons  = da['longitude'][:]
    lats  = da['latitude'][:]
    clm   = da.groupby('time.month').mean(dim='time')
    anm   = (da.groupby('time.month') - clm)
    vstr  = da.name
    vunt  = da.units

    time  = asl_df['time']
    xtime = np.linspace(1, len(time), len(time))
    asl_df = asl_df.set_index('time')
    asl_exist = False
    for col in colnams:
        if 'RelCenPres' in col:
            asl = asl_df[col].to_xarray()
            asl_exist = True
    if not asl_exist:
        exit('RelCenPres not exist, please check....')

    asl   = asl.assign_coords({"time": da.time})
    aslSD = asl / asl.std(dim='time')
    years = int(time[0].split("-")[0])
    yeare = int(time[len(time)-1].split("-")[0])

    raslSD    = aslSD.copy()
    ranm      = anm.copy()
    raslSD[:] = low_pass(1.0/5.0, aslSD, axis=0)
    if np.isnan(anm).any():
        tmp1 = ranm.fillna(-99999)
        tmp1 = low_pass(1.0/5.0, tmp1, axis=0)
        ranm = anm.copy()
        ranm[:, :, :] = tmp1[:, :, :]
        ranm = ranm.where(ranm > -10000)
        del tmp1
    else:
        ranm[:, :, :] = low_pass(1.0/5.0, anm[:, :, :], axis=0)
    rdanm = detrend_dim(ranm, 'time', 1)

    vcor = xr.corr(raslSD, rdanm, dim="time")
    vreg = xr.cov(raslSD, rdanm, dim="time") / raslSD.var(dim='time', skipna=True).values
    # Replaces xs.pearson_r_p_value(raslSD, rdanm, dim="time", skipna=True)
    pval = pearson_r_p_value(raslSD, rdanm, dim="time")

    vsig = vreg.copy()
    vsig = vsig.where(pval <= 0.05)

    save_figure_data(vcor, vreg, pval, vsig, mip, exp, relm, case, case_id, period, var, vunt, out_path)

    vmin  = -1.0
    vmax  =  1.0
    nlev  =  21
    fontsize = 16
    bartitle = 'Regressed {}({})'.format(vstr, vunt)
    fig  = plt.figure(figsize=(10, 12))
    grid = fig.add_gridspec(ncols=1, nrows=1)
    ax1, fill1 = draw_regression_map(vstr, vunt, lons, lats, vcor, vreg, pval, fig, asl_region,
                                     fontsize, 'SAM-PSL Pattern', grid[0, 0], vmin, vmax, nlev)
    cb = fig.colorbar(fill1, ax=[ax1], drawedges=True, orientation='horizontal',
                      shrink=0.95, aspect=40, pad=0.05, extendfrac='auto', extendrect=True)
    ticks  = np.linspace(vmin, vmax, nlev)
    labels = []
    for i, tick in enumerate(ticks):
        if i % 2 == 0:
            labels.append('{:0.1f}'.format(tick))
        else:
            labels.append('')
    cb.set_ticks(ticks=ticks, labels=labels, fontsize=fontsize * 0.9)
    cb.set_label(label=bartitle, fontsize=fontsize * 0.95)
    plt.rcParams["font.family"] = "sans-serif"
    plt.rcParams.update({'font.size': fontsize})
    plt.draw()
    if not os.path.exists(fig_path):
        os.makedirs(fig_path)
    fig_name = "fig_2d_map_{}_{}_{}_{}_{}_{}_{}.pdf".format(mip, exp, case, relm, case_id, var, period)
    plt.savefig(os.path.join(fig_path, fig_name))
    plt.close()
    return


# ============================================================
# Case class
# ============================================================
class Case:
    def __init__(self, path, var, color, label):
        self._path  = path
        self._color = color
        self._label = label
        self._var   = var

    @property
    def path(self):
        return self._path

    @property
    def color(self):
        return self._color

    @property
    def label(self):
        return self._label


# ============================================================
# Main analysis function
# ============================================================
def main(fig_path, out_path, mip, exp, relm, case_id, period, case_dict, asl_region,
         asl_min_dist, asl_num_peak, asl_exc_bord, l_check_asl_region, l_allow_no_asl):
    for key in case_dict:
        if key == "mask":
            dmsk = case_dict[key].path
            vmsk = case_dict[key]._var
        else:
            case = key
            var  = case_dict[key]._var
            data = case_dict[key].path

    print("working on ", case, var)

    ds   = xr.open_dataset(data)
    ymds = '{}-{}-01'.format(period.split("-")[0][0:4], period.split("-")[0][4:6])
    ymde = '{}-{}-31'.format(period.split("-")[1][0:4], period.split("-")[1][4:6])
    ds   = ds.sel(time=slice(ymds, ymde))
    if len(ds.dims) < 3:
        print("data dimension is incorrect")
        exit()
    else:
        if ds[var].dims[1] == "lat" or ds[var].dims[2] == "lon":
            ds = ds.rename({ds[var].dims[1]: 'latitude',
                            ds[var].dims[2]: 'longitude'})

    if os.path.exists(dmsk):
        dsm  = xr.open_dataset(dmsk)
        if dsm[vmsk].dims[0] == "lat" or dsm[vmsk].dims[1] == "lon":
            dsm = dsm.rename({dsm[vmsk].dims[0]: 'latitude',
                              dsm[vmsk].dims[1]: 'longitude'})
        mask = dsm[vmsk]
        mask = mask / 100.0  # range [0, 1]
    else:
        print("Warning: land/sea mask not exist, derive it...")
        lons_tmp = ds.longitude.copy()
        lats_tmp = ds.latitude
        if lons_tmp.values.min() > -1:
            lons_tmp = (lons_tmp + 180) % 360 - 180
        # Use regionmask to derive land/sea mask (replaces global_land_mask)
        land_110 = regionmask.defined_regions.natural_earth_v5_0_0.land_110
        mask_da  = land_110.mask(lons_tmp.values, lats_tmp.values)
        # 1 = land, 0 = ocean  (consistent with globe.is_land convention)
        mask = (~np.isnan(mask_da.values)).astype(float)
        del lons_tmp, lats_tmp, mask_da

    da   = ds[var]
    if da.units == "Pa":
        print("change units: from ", da.units, " to ", "hPa")
        da = da / 100.
        da = da.assign_attrs(units='hPa')
    vstr = da.name.upper()
    vunt = da.units

    if l_check_asl_region:
        da_t     = da.sel(time=da.time[0:11])
        da_mask2 = da_t.where(mask == 0)
        da_nmsk  = slice_region(da_t, asl_region)
        da_mask2 = slice_region(da_mask2, asl_region)
        print(da_mask2)
        print("before mask,min/max: ", da.min(), da.max())
        print("after mask,min/max: ", da_mask2.min(), da_mask2.max())
        plot_regions_mask(fig_path, da_nmsk, da_mask2, case)
        del da_t, da_mask2, da_nmsk

    times = da.time.dt.strftime("%Y-%m-%d")
    ntime = len(times)
    all_lows_dfs = pd.DataFrame()
    print("number of total months in data: ", ntime)
    for t in range(ntime):
        da_t         = da.isel(time=t)
        da_mask2     = da_t.where(mask == 0)
        da_mask2     = slice_region(da_mask2, asl_region)
        all_lows_df  = get_lows(da_mask2, asl_region, asl_min_dist, asl_num_peak, asl_exc_bord)
        all_lows_dfs = pd.concat([all_lows_dfs, all_lows_df], ignore_index=True)
        del da_t, all_lows_df

    asl_df = pd.DataFrame()
    asl_df, colnams = define_asl(times, all_lows_dfs, asl_region, l_allow_no_asl,
                                  mip, exp, relm, case, case_id, period, vstr, out_path)

    draw_asl_loc(da, mask, asl_df, asl_region, mip, exp, relm, case, case_id, period, vstr, vunt, fig_path)
    draw_asl_ts(fig_path, asl_df, colnams, asl_region, mip, exp, relm, case, case_id, period, vstr, vunt)
    draw_asl_map(out_path, fig_path, ds[var], asl_df, colnams, asl_region, mip, exp, relm,
                 case, case_id, period, vstr, vunt)
    return


In [ ]:
if __name__ == "__main__":

  top_path = "/lcrc/group/e3sm2/ac.dcomeau/E3SMv3_dev/v3.LR.piControl-scaled-dismf" 
  out_path = os.path.join(top_path,"E3SMv21_testings","paper_material","fig_data","asl_analysis","raw_index")
  fig_path = os.path.join(top_path,"E3SMv21_testings","paper_material","3_asl_analysis","asl_index_ts","figure")

  # region of interest (asl sector)
  asl_region   = {'west':170., 'east':298., 'south':-80., 'north':-60.}
  # tunable parameters for ASL search. They need to be adjusted with the model
  # resolution and if you want to have non-missing ASLs. 
  asl_min_dist = 5 # peaks are separated by at least min_distance
  asl_num_peak = 3 # maximum number of peaks
  asl_exc_bord = False # excludes peaks from within min_distance pixels of the border

  #sanity check (plot region with and without mask) 
  l_check_asl_region = False #True  
  #if asl is not identified in the selected region, then search outside region 
  #and find a loction with local minimum pressure when l_allow_no_asl = False.
  # if l_allow_no_asl = True, then missing values will be asigned if asl is not found  
  l_allow_no_asl = False #True 

  mip       = "e3sm"
  exps      = [ "piControl"]
  relms     = [ "1950"]
  products  = ["v2_1-SORRM"]
  tableId   = "Amon"
  var       = "psl"
  varstr    = "PSL"
  case_id   = 'asl_scotthoskingv3'
  period    = "080101-100012"
  run_path  = "/lcrc/group/e3sm/ac.szhang/acme_scratch/data/pcmdi/model/monthly"
  run_mask  = "/lcrc/group/e3sm/ac.szhang/acme_scratch/data/pcmdi/model/fixed/sftlf"

  for exp in exps:
    for relm in relms:
      for product in products:
        case = product
        ptmp = os.path.join(run_path,mip,exp,tableId,var)
        ftmp = '{}.{}.{}.{}.*.{}.{}.nc'.format(mip,exp,product,relm,var,period)
        filePath = sorted(glob.glob(os.path.join(ptmp,ftmp)))
        if len(filePath) > 0 and os.path.isfile(filePath[0]):
          print("case: ", case)
          print("data: ", filePath[0])
          fileName = filePath[0].split("/")[-1]
          asl_min_dist = 1  # peaks are separated by at least min_distance
          asl_num_peak = 12 # maximum number of peaks
          data_fil = filePath[0]
          mask_fil = os.path.join(run_mask,tableId,mip+"."+exp+"."+product+".fx.sftlf.nc")

          case_dict = collections.OrderedDict()
          case_dict[case] = Case(data_fil, var= var.upper(), color="blue", label=case)
          case_dict['mask'] = Case(mask_fil, var= "sftlf", color="blue", label=case)

          #call fuction to generate ASL index 
          main(fig_path,out_path,mip,exp,relm,case_id,period,case_dict,asl_region,asl_min_dist, 
               asl_num_peak,asl_exc_bord,l_check_asl_region,l_allow_no_asl)


case:  v2_1-SORRM
data:  /lcrc/group/e3sm/ac.szhang/acme_scratch/data/pcmdi/model/monthly/e3sm/piControl/Amon/psl/e3sm.piControl.v2_1-SORRM.1950.mon.psl.080101-100012.nc
working on  v2_1-SORRM PSL
change units: from  Pa  to  hPa
number of total months in data:  2400
